# PLIF Recall Analysis

Two datasets:
- `ALL_1_poses_plif_datesplit_plif0.5_combined_results.csv` — evaluator bootstrap results: fraction of ligands with PLIF Tversky recall ≥ 0.5, scored by RMSD or POSIT, DateSplit vs RandomSplit
- `plif_recall_combined.csv` — per-(compound, reference) raw PLIF recall values for distribution inspection

## Imports

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from plot_style import (
    FONT_SIZES, LARGE_FIG_SIZE, SMALL_FIG_SIZE, ALPHA, Y_TICKS,
    LABEL_MAP, apply_style, save_fig_both, plot_filled_in_error_bars,
)
apply_style()

## Paths

In [ ]:
RESULTS_DIR = Path("/Users/apayne/Downloads/full_cross_dock_v2/analyzed_results")

PLIF_EVALUATOR_CSV   = RESULTS_DIR / "ALL_1_poses_plif_datesplit_plif0.5_combined_results.csv"
PLIF_RECALL_CSV      = RESULTS_DIR / "plif_recall_combined.csv"
RMSD_EVALUATOR_CSV   = RESULTS_DIR / "ALL_1_poses_datesplit_combined_results.csv"  # for comparison

## Load evaluator results

In [ ]:
plif_df = pd.read_csv(PLIF_EVALUATOR_CSV)
plif_df.shape

In [ ]:
# Unique values in key columns
for col in ["Score", "EvaluationMetric", "EvaluationMetric_Cutoff", "Reference_Split", "N_Reference_Structures"]:
    print(f"{col}: {sorted(plif_df[col].dropna().unique().tolist())}")

## Apply label map

In [ ]:
# Extend with PLIF-specific labels
label_map = {
    **LABEL_MAP,
    "PLIFData_plif_tversky_recall": "PLIF Tversky Recall",
    "Fraction": "Fraction of Ligands with\nPLIF Recall ≥ 0.5",
}

df = plif_df.rename(columns=label_map)
for col in df.columns:
    df[col] = df[col].map(lambda x: label_map.get(x, x))

df.head(2)

In [ ]:
n_refs = sorted(plif_df["N_Reference_Structures"].unique().tolist())
print("N reference structures:", n_refs)

# PLIF Recall ≥ 0.5 success rate vs N reference structures

In [ ]:
fig = plot_filled_in_error_bars(
    df,
    x_var=label_map["N_Reference_Structures"],
    y_var=label_map["Fraction"],
    color_var=label_map["Reference_Split"],
    style_var=label_map["Score"],
    ci_lower=label_map["CI_Lower"],
    ci_upper=label_map["CI_Upper"],
    n_refs=n_refs,
    reverse_hue_order=True,
)
save_fig_both(fig.gcf(), "plif_recall_vs_n_references")

# Comparison: PLIF recall vs RMSD as the evaluation metric

Overlay DateSplit curves for both metrics to see how correlated the two success rates are.

In [ ]:
rmsd_df = pd.read_csv(RMSD_EVALUATOR_CSV)

# Add a column distinguishing which metric is being evaluated
plif_df["Metric"] = "PLIF Recall ≥ 0.5"
rmsd_df["Metric"] = "RMSD < 2 Å"

# Keep only DateSplit rows and POSIT scoring for a clean 2-line comparison
compare_df = pd.concat([plif_df, rmsd_df], ignore_index=True)
compare_df = compare_df[compare_df["Reference_Split"] == "DateSplit"]

compare_label_map = {
    **label_map,
    "Metric": "Evaluation Metric",
    "Fraction": "Success Rate",
}
cdf = compare_df.rename(columns=compare_label_map)
for col in cdf.columns:
    cdf[col] = cdf[col].map(lambda x: compare_label_map.get(x, x))

n_refs_compare = sorted(compare_df["N_Reference_Structures"].unique().tolist())

fig = plot_filled_in_error_bars(
    cdf,
    x_var=compare_label_map["N_Reference_Structures"],
    y_var=compare_label_map["Fraction"],
    color_var=compare_label_map["Metric"],
    style_var=compare_label_map["Score"],
    ci_lower=compare_label_map["CI_Lower"],
    ci_upper=compare_label_map["CI_Upper"],
    n_refs=n_refs_compare,
)
save_fig_both(fig.gcf(), "plif_vs_rmsd_datesplit_comparison")

# Distribution of raw PLIF Tversky recall values

Per-(compound, reference) values from `plif_recall_combined.csv`. Shows where the 0.5 cutoff sits in the distribution.

In [ ]:
recall_df = pd.read_csv(PLIF_RECALL_CSV)
print(recall_df.shape)
recall_df.head(3)

In [ ]:
fig, ax = plt.subplots(figsize=SMALL_FIG_SIZE)

ax.hist(recall_df["plif_tversky_recall"].dropna(), bins=40, color="steelblue", edgecolor="none", alpha=0.8)
ax.axvline(0.5, color="firebrick", lw=2, linestyle="--", label="Cutoff (0.5)")

ax.set_xlabel("PLIF Tversky Recall", fontsize=FONT_SIZES["xlabel"], fontweight="bold")
ax.set_ylabel("Count", fontsize=FONT_SIZES["ylabel"], fontweight="bold")
ax.tick_params(axis="both", labelsize=FONT_SIZES["ticks"])
legend = ax.legend()
plt.setp(legend.get_texts(), fontsize=FONT_SIZES["legend_text"])
sns.despine()
plt.tight_layout()
save_fig_both(fig, "plif_recall_distribution")

## Fraction of pairs above cutoff

In [ ]:
frac_above = (recall_df["plif_tversky_recall"] >= 0.5).mean()
print(f"Fraction of (compound, reference) pairs with PLIF recall ≥ 0.5: {frac_above:.3f}")

## Per-compound best PLIF recall across all references

Useful sanity check: what fraction of compounds can achieve recall ≥ 0.5 against *at least one* reference?

In [ ]:
best_per_compound = recall_df.groupby("compound_name")["plif_tversky_recall"].max()
frac_achievable = (best_per_compound >= 0.5).mean()
print(f"Fraction of compounds with best PLIF recall ≥ 0.5 against any reference: {frac_achievable:.3f}")

fig, ax = plt.subplots(figsize=SMALL_FIG_SIZE)
ax.hist(best_per_compound.dropna(), bins=40, color="steelblue", edgecolor="none", alpha=0.8)
ax.axvline(0.5, color="firebrick", lw=2, linestyle="--", label="Cutoff (0.5)")
ax.set_xlabel("Best PLIF Tversky Recall (any reference)", fontsize=FONT_SIZES["xlabel"], fontweight="bold")
ax.set_ylabel("Number of compounds", fontsize=FONT_SIZES["ylabel"], fontweight="bold")
ax.tick_params(axis="both", labelsize=FONT_SIZES["ticks"])
legend = ax.legend()
plt.setp(legend.get_texts(), fontsize=FONT_SIZES["legend_text"])
sns.despine()
plt.tight_layout()
save_fig_both(fig, "best_plif_recall_per_compound")